# Leave-one-room-out test of the P13 network

Trains the P13 architecture from scratch with whole rooms or single recordings held out (`temporal_model/loro.py`).
Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). About 20 minutes in total, most of it the 12 training runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Clone the code (public repository, `main` branch), or update an existing clone.

In [ ]:
import os, subprocess
if not os.path.exists('/content/Huawei'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Agata872/Huawei.git', '/content/Huawei'], check=True)
%cd /content/Huawei
!git pull -q
!git log --oneline -1
!pip install -q 'blosc2>=3,<5'

Download the labeled archive (2.1 GB) from the link in `FutureNetwork-starting-kit/data_downloads.md`, check its SHA-256 and extract it.

In [ ]:
%%bash
set -e
mkdir -p data/downloads data/extracted
cd data/downloads
[ -f FutureNetwork-data-labeled.zip ] || wget -q https://dl.r0h.eu/techarena2026-paris/FutureNetwork-data-labeled.zip
echo '1f60df2d4c1459c307e444741056e17a5ec537614cf577331ed04eb94d0765c1  FutureNetwork-data-labeled.zip' | sha256sum -c
unzip -q -n FutureNetwork-data-labeled.zip -d ../extracted
ls ../extracted/labeled | wc -l

Precompute the block maps (CPU, a few minutes) and check them against the raw-input features: the printed difference should be ~1e-6 or smaller.

In [ ]:
!python -m temporal_model.maps --check

Train and score: c1 and a1 held out as whole rooms, and one recording per headcount held out inside each room, three seeds each.

In [ ]:
!python -m temporal_model.loro --holdout room:c1,room:a1,traces:c1,traces:a1 --seeds 0,1,2

Zip the per-window results (logits included) for the local repository's `runs/temporal_model/`. The browser download only works in the Colab web UI; elsewhere fetch `/content/loro_results.zip` from the runtime's file browser.

In [ ]:
!cd runs && zip -qr /content/loro_results.zip temporal_model
try:
    from google.colab import files
    files.download('/content/loro_results.zip')
except Exception as error:
    print(f'browser download unavailable ({error}); the zip is at /content/loro_results.zip')